## Imports

In [ ]:
import json
import base64
import urllib.request
from pathlib import Path

## Configuration

In [ ]:
ENDPOINT_URL = "http://localhost:8080/invocations"
IMAGE_DIR = Path.home() / "Downloads" / "hawaii-camera-traps"
ADDAX_JSON = IMAGE_DIR / "image_recognition_file.json"

## Load AddaxAI Results

In [ ]:
with open(ADDAX_JSON) as f:
    addax_data = json.load(f)

cls_cats = addax_data.get("classification_categories", {})

print(f"Images: {len(addax_data['images'])}")
print(f"Classification categories: {json.dumps(cls_cats, indent=2)}")

## Verify Endpoint Health

In [ ]:
ping_url = ENDPOINT_URL.replace("/invocations", "/ping")
resp = urllib.request.urlopen(ping_url, timeout=10)
print(json.loads(resp.read()))

## Run Comparison

In [ ]:
results = []

for img in addax_data["images"]:
    for det in img.get("detections", []):
        cls = det.get("classifications", [])
        if not cls:
            continue

        # Read and encode image
        with open(IMAGE_DIR / img["file"], "rb") as f:
            img_b64 = base64.b64encode(f.read()).decode()

        # Call our endpoint
        payload = json.dumps({"image": img_b64, "bbox": det["bbox"]}).encode()
        req = urllib.request.Request(ENDPOINT_URL, data=payload, headers={"Content-Type": "application/json"})
        resp = urllib.request.urlopen(req, timeout=300)
        preds = json.loads(resp.read())

        our_top = max(preds, key=preds.get)
        our_conf = preds[our_top]

        addax_top_id, addax_conf = cls[0]
        addax_top = cls_cats.get(str(addax_top_id), str(addax_top_id))

        match = our_top == addax_top
        conf_match = abs(our_conf - addax_conf) < 0.0001

        results.append({
            "file": img["file"],
            "addax_class": addax_top,
            "addax_conf": addax_conf,
            "our_class": our_top,
            "our_conf": our_conf,
            "class_match": match,
            "conf_match": conf_match,
        })

print(f"Processed {len(results)} detections")

## Results

In [ ]:
class_matches = sum(1 for r in results if r["class_match"])
conf_matches = sum(1 for r in results if r["conf_match"])

print(f"Class match: {class_matches}/{len(results)}")
print(f"Confidence match (±0.0001): {conf_matches}/{len(results)}")
print()

for r in results:
    icon = "✅" if r["class_match"] and r["conf_match"] else "❌"
    print(f"{icon} {r['file']}")
    print(f"   AddaxAI: {r['addax_class']} ({r['addax_conf']:.4f})")
    print(f"   Ours:    {r['our_class']} ({r['our_conf']:.4f})")
    print()